In [ ]:

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver


from selenium.webdriver.common.by import By

from time import sleep

import os

from odf.table import Table

from odf.opendocument import load
from odf.text import P

from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P
import csv

In [ ]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'TW SFBTW' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running TW SFBTW Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'TW SFBTW 1':'Securities Firms' , 

        'TW SFBTW 2': 'Futures Enterprises', 

        'TW SFBTW 3': 'Securities Investment Trust Enterprises',

        'TW SFBTW 4': 'Securities Investment Consulting Enterprises', 

        'TW SFBTW 5': 'Securities Finance Enterprises', 

        'TW SFBTW 6': 'Credit Rating Agencies', 


        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}


processdate = now.strftime('%Y-%m-%d')





In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict





In [6]:
driver.get('https://www.sfb.gov.tw/en/home.jsp?id=140')

tables_info = []
for k, reg in enumerate(regdict):

	print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} : {regdict[reg]}")

	div_element = driver.find_element(By.XPATH, f"//div[contains(text(), '{regdict[reg]}')]")

	links = div_element.find_elements(By.TAG_NAME, 'a')

	document_link = ''

	if len(links[0].get_attribute("href"))>10:
		sleep(3)
		driver.get(links[-1].get_attribute("href"))
		document_link = links[-1].get_attribute("href")
	else:
		sleep(3)
		driver.get(links[-1].get_attribute('href'))
		document_link = links[-1].get_attribute("href")
	sleep(3)
	dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
	sleep(3)
	
	if dl_files[0].endswith('ods'):
		df_list_ods = pd.read_excel(dl_files[0], engine="odf")
		df_list_ods.columns = df_list_ods.iloc[0]
		df_list_ods = df_list_ods[1:]

		sqldict['Name'].extend(df_list_ods[df_list_ods.columns[1]].tolist())
		sqldict['Phone'].extend(df_list_ods[df_list_ods.columns[4]].tolist())
		sqldict['Website'].extend(df_list_ods[df_list_ods.columns[2]].tolist())
		sqldict['Address_1'].extend(df_list_ods[df_list_ods.columns[3]].tolist())

		addresses = df_list_ods[df_list_ods.columns[3]].tolist()
		cntries, cities, zips = [], [], []
		for addr in addresses:
			cntry = addr.split(',')[-1] 
			city_zip = addr.split(',')[-2] 
			zip_code = city_zip.split(' ')[-1] 
			city = city_zip.replace(zip_code, '').strip()
			cntries.append(cntry)
			cities.append(city)
			zips.append(zip_code)

		sqldict['Cntry'].extend(cntries)
		sqldict['City'].extend(cities)
		sqldict['Zip'].extend(zips)

		n = len(df_list_ods[df_list_ods.columns[1]].tolist())

		sqldict['ListProcessDate'].extend([processdate] * n)
		sqldict['RegCtry'].extend([reg.split(' ')[0]] * n)
		sqldict['RegCode'].extend([reg.split(' ')[1]] * n)
		sqldict['ListCode'].extend([reg.split(' ')[-1]] * n)
		sqldict['ListName'].extend([regdict[reg]] * n)
		sqldict['RegulationType'].extend(['Regulated'] * n)


		sqldict = bourange_same_length_array(sqldict)





	else:
		sleep(3)
		textdoc = load(dl_files[0])

		all_tables = textdoc.getElementsByType(Table)

		for table in all_tables:
			tables_info.append(table)

		# Prepare to write to CSV
		with open(f"output+{reg}.csv", "w", newline="", encoding="utf-8") as csvfile:
			writer = csv.writer(csvfile)

			for table in all_tables:
				rows = table.getElementsByType(TableRow)
				for row in rows:
					cells = row.getElementsByType(TableCell)
					row_data = []
					for cell in cells:
						cell_text = ""
						for p in cell.getElementsByType(P):
							cell_text += str(p) 
						row_data.append(cell_text.strip())
					writer.writerow(row_data)
		sleep(3)

		df = pd.read_csv(f'output+{reg}.csv')
		for idx, row in df.iterrows():
			if row.iloc[0] == df.columns[0]:
				continue
			data_list = row.tolist()
			# print(data_list[0])
			# print(data_list[1])
			sqldict['Name'].append(data_list[1])
			sqldict['InternalID_1'].append(data_list[0])
			sqldict['InternalID_1_type'].append(df.columns[0])
			sqldict['RegulationType'].append('Regulated')
			sqldict['ListProcessDate'].append(processdate)
			sqldict['RegCtry'].append(reg.split(' ')[0]) 
			sqldict['RegCode'].append(reg.split(' ')[1])
			sqldict['ListCode'].append(reg.split(' ')[-1])
			sqldict['ListName'].append(regdict[reg])
			sqldict = bourange_same_length_array(sqldict)


	

	if os.path.exists(tempfolder):

		for rem in os.listdir(tempfolder):

			os.remove(os.path.join(tempfolder, rem))

	else:

		os.mkdir(tempfolder)

	

    

[INFO] : Working 1/6 | TW SFBTW 1 : Securities Firms
[INFO] : Working 2/6 | TW SFBTW 2 : Futures Enterprises
[INFO] : Working 3/6 | TW SFBTW 3 : Securities Investment Trust Enterprises
[INFO] : Working 4/6 | TW SFBTW 4 : Securities Investment Consulting Enterprises
[INFO] : Working 5/6 | TW SFBTW 5 : Securities Finance Enterprises
[INFO] : Working 6/6 | TW SFBTW 6 : Credit Rating Agencies


In [8]:
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

In [35]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_29976\3641028257.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


ValueError: Invalid file path or buffer object type: <class '_csv.writer'>

In [10]:
filename

'TW SFBTW SQL Read 2025-11-03 09.26.41.xlsx'

In [36]:
df.to_csv('total.csv')